# HealthConnect Clinic: Week 7 KPI & Finding Testing
**Track:** Data Analytics
**Author:** Hillary Emmanuel
**Programme:** AnalystLab Africa Experience Lab

This notebook reproduces, independently of the Week 6 notebook and the Power BI dashboard, the
Week 7 testing pass: KPI reconciliation (all five KPIs), segment robustness checks on the lead-time
finding, a confidence-interval overlap check and a lead-time redundancy check on the distance-band
finding, and the investigation/correction of the double-risk segment definition, including the
cross-track check against the DS track collaborator's `double_risk_flag` feature (retested with both
the Random Forest and the exact Week 6 candidate Logistic Regression; the exact-candidate test is in `scripts/week7_double_risk_exact_candidate_test.py`, and Section 5b is an earlier preliminary check).

Reads `../data/processed/healthconnect_appointments_prepared.csv` and the four Week 6 `kpi*_ci_*.csv`
tables read-only. Does not modify source data. Needs scikit-learn for Sections 3b, 5 and 5b.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/healthconnect_appointments_prepared.csv")
y = df["is_no_show"].astype(int)
print(f"Rows: {len(df)}")
df.head()

Rows: 5000


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,...,waiting_time_minutes,appointment_outcome,is_no_show,distance_known,prior_no_show_bracket,lead_time_band,distance_band,lead_time_band_order,distance_band_order,prior_no_show_bracket_order
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2025-02-06,2025-02-18,Tuesday,Afternoon,...,29.0,No-Show,True,True,0,8-14 days,15+ km,2,4.0,1
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2026-02-25,2026-02-27,Friday,Morning,...,42.0,Attended,False,True,0,0-7 days,10-15 km,1,3.0,1
2,HC-00003,P-1366,Female,50,45-54,General Consultation,2025-11-16,2025-12-24,Wednesday,Morning,...,11.0,No-Show,True,True,1,31-45 days,10-15 km,4,3.0,2
3,HC-00004,P-1031,Male,59,55-64,Follow-up,2025-07-18,2025-08-28,Thursday,Evening,...,35.0,Attended,False,True,1,31-45 days,5-10 km,4,2.0,2
4,HC-00005,P-1458,Female,34,25-34,Follow-up,2025-07-09,2025-08-25,Monday,Afternoon,...,27.0,No-Show,True,True,1,46-60 days,5-10 km,5,2.0,2


## 1. KPI Reconciliation — Dashboard vs Source CSVs vs Independent Recompute

For each KPI, compare the Week 6 CI table's point estimate against a fresh groupby computed directly from the raw prepared dataset.

In [2]:
kpi_checks = {
    "prior_no_show_bracket": "../data/processed/kpi1_ci_prior_no_show_bracket.csv",
    "lead_time_band": "../data/processed/kpi2_ci_lead_time_band.csv",
    "distance_band": "../data/processed/kpi3_ci_distance_band.csv",
    "reminder_sent": "../data/processed/kpi4_ci_reminder_sent.csv",
}

for col, path in kpi_checks.items():
    ci = pd.read_csv(path)
    recompute = df.groupby(col)["is_no_show"].mean() * 100
    print(f"=== {col} ===")
    print(ci)
    print("\nIndependent recompute (%):")
    print(recompute.round(2))
    print()

=== prior_no_show_bracket ===
   prior_no_show_bracket  PointEstimate    CI_Low   CI_High
0                      0       0.435125  0.415945  0.453954
1                      1       0.534884  0.509690  0.560724
2                      2       0.593607  0.545662  0.634703
3                      3       0.688172  0.580645  0.784946

Independent recompute (%):
prior_no_show_bracket
0     43.51
1     53.49
2     59.36
3+    68.82
Name: is_no_show, dtype: float64

=== lead_time_band ===
  lead_time_band  PointEstimate    CI_Low   CI_High
0       0-7 days       0.278125  0.245273  0.312539
1      8-14 days       0.335548  0.298962  0.375415
2     15-30 days       0.432108  0.405851  0.457614
3     31-45 days       0.538156  0.509519  0.565183
4     46-60 days       0.676949  0.648672  0.702678

Independent recompute (%):
lead_time_band
0-7 days      27.81
15-30 days    43.21
31-45 days    53.82
46-60 days    67.69
8-14 days     33.55
Name: is_no_show, dtype: float64

=== distance_band ===
  di

**Result: PASS on all four KPIs.** Point estimates in the Week 6 CI tables match an independent recomputation from the raw prepared dataset exactly (to display precision), and match the dashboard's displayed values (visually confirmed against the exported dashboard image). This confirms the dashboard is not dependent on any notebook-only logic and is reproducible directly from source data.

## 1b. KPI 5: Cancellation-to-No-Show Ratio

KPI 5 is charted on the dashboard (bottom right, by lead-time band, no error bars) but was not covered by the
reconciliation above. It is recomputed here from the prepared dataset and compared with the Week 6
KPI Validation report. The bootstrap uses a fresh random seed, so interval bounds can differ from
Week 6 by a few thousandths; the point estimates are what get compared.

In [ ]:
df["canc"] = df["appointment_outcome"].str.contains("Cancel", case=False)
df["ns"] = df["is_no_show"].astype(bool)
rng = np.random.default_rng(42)

def ratio_ci(d, B=2000):
    a = np.column_stack([d["canc"].values, d["ns"].values]); n = len(a); v = []
    for _ in range(B):
        s = a[rng.integers(0, n, n)]
        v.append(s[:, 0].sum() / s[:, 1].sum())
    return d["canc"].sum() / d["ns"].sum(), *np.percentile(v, [2.5, 97.5])

# Week 6 KPI_Validation report values: (point estimate, CI low, CI high)
week6 = {"Overall": (0.109, 0.095, 0.122), "0-7 days": (0.202, 0.138, 0.286), "8-14 days": (0.139, 0.088, 0.197),
         "15-30 days": (0.118, 0.090, 0.148), "31-45 days": (0.105, 0.079, 0.132), "46-60 days": (0.076, 0.057, 0.097)}
groups = {"Overall": df}
for b in ["0-7 days", "8-14 days", "15-30 days", "31-45 days", "46-60 days"]:
    groups[b] = df[df["lead_time_band"] == b]

print(f"{'Group':11s} {'n':>5s} {'Week 7':>7s} {'Week 6':>7s}  {'Week 7 CI':17s} {'Week 6 CI':17s} Match")
for name, d in groups.items():
    est, lo, hi = ratio_ci(d); w = week6[name]
    print(f"{name:11s} {len(d):5d} {est:7.3f} {w[0]:7.3f}  [{lo:.3f}, {hi:.3f}]".ljust(52) + f"[{w[1]:.3f}, {w[2]:.3f}]".ljust(18) + ("PASS" if abs(est - w[0]) < 0.0005 else "CHECK"))

Group           n  Week 7  Week 6  Week 7 CI         Week 6 CI         Match
Overall      5000   0.109   0.109  [0.095, 0.124][0.095, 0.122]    PASS
0-7 days      640   0.202   0.202  [0.134, 0.279][0.138, 0.286]    PASS
8-14 days     602   0.139   0.139  [0.089, 0.200][0.088, 0.197]    PASS
15-30 days   1333   0.118   0.118  [0.090, 0.151][0.090, 0.148]    PASS
31-45 days   1258   0.105   0.105  [0.080, 0.132][0.079, 0.132]    PASS
46-60 days   1167   0.076   0.076  [0.057, 0.098][0.057, 0.097]    PASS


**Result: PASS.** All six point estimates match the Week 6 report. The ratio falls from about 0.20 at
0-7 days to about 0.08 at 46-60 days, and the intervals for those two ends do not overlap: as lead
time grows, cancellations become a smaller share compared with no-shows. Week 6's reading that long
lead-time bookings fail silently, not through cancellations, holds up. All five KPIs are now covered.

## 2. Lead-Time Finding — Segment Robustness Test

Week 6 established booking lead time as the strongest predictor. Testing here checks whether the finding holds across two segments not previously tested: `appointment_type` and `age_group`.

In [3]:
order = ["0-7 days", "8-14 days", "15-30 days", "31-45 days", "46-60 days"]

print("No-show rate by lead_time_band x appointment_type:")
pivot1 = df.pivot_table(index="lead_time_band", columns="appointment_type", values="is_no_show", aggfunc="mean").reindex(order)
display(pivot1.round(3))

print("\nNo-show rate by lead_time_band x age_group:")
pivot2 = df.pivot_table(index="lead_time_band", columns="age_group", values="is_no_show", aggfunc="mean").reindex(order)
display(pivot2.round(3))

No-show rate by lead_time_band x appointment_type:


appointment_type,Diagnostic Test,Follow-up,General Consultation,Specialist Consultation
lead_time_band,,,,
0-7 days,0.297,0.294,0.246,0.306
8-14 days,0.319,0.373,0.337,0.283
15-30 days,0.462,0.429,0.421,0.445
31-45 days,0.491,0.601,0.518,0.519
46-60 days,0.714,0.704,0.649,0.670



No-show rate by lead_time_band x age_group:


age_group,18-24,25-34,35-44,45-54,55-64,65+
lead_time_band,,,,,,
0-7 days,0.386,0.240,0.208,0.280,0.287,0.291
8-14 days,0.375,0.353,0.396,0.340,0.346,0.261
15-30 days,0.483,0.482,0.428,0.423,0.427,0.393
31-45 days,0.493,0.526,0.553,0.533,0.536,0.566
46-60 days,0.662,0.726,0.681,0.661,0.729,0.621


**Result: PASS.** The 46–60 day lead-time band is the highest no-show-rate band in every `appointment_type` category and every `age_group` category. The finding is robust across both new segments.

## 3. Distance-Band Finding — Confidence Interval Overlap Check

Week 6 downgraded the "15km cliff" finding based on bootstrap CIs. This re-checks that the three middle bands genuinely overlap, and that only the 15+km band is distinct.

In [4]:
kpi3 = pd.read_csv("../data/processed/kpi3_ci_distance_band.csv")
display(kpi3)

print("\nConsecutive-band CI overlap check:")
for i in range(len(kpi3) - 1):
    a, b = kpi3.iloc[i], kpi3.iloc[i + 1]
    overlap = a["CI_High"] >= b["CI_Low"]
    print(f"{a['distance_band']:>10} vs {b['distance_band']:<10}: CI overlap = {overlap}")

,distance_band,PointEstimate,CI_Low,CI_High
0,<5 km,0.464533,0.436830,0.491349
1,5-10 km,0.465130,0.440884,0.489362
2,10-15 km,0.484982,0.458458,0.515018
3,15+ km,0.540860,0.508602,0.574194



Consecutive-band CI overlap check:
     <5 km vs 5-10 km   : CI overlap = True
   5-10 km vs 10-15 km  : CI overlap = True
  10-15 km vs 15+ km    : CI overlap = True


**Result: PASS.** All three consecutive middle-band CI pairs overlap, confirming they are statistically indistinguishable. Only the 15+km band sits outside this overlapping cluster. This confirms the Week 6 text downgrade is correct — the chart itself still needs a visual refinement to communicate this (tracked separately, not a notebook task).

## 3b. Distance vs Lead Time: Is the Distance Effect Redundant?

Week 6 flagged an open question: does distance overlap with lead time, which would explain why the
model gives distance little weight? Three checks: (1) are distance and lead time related at all,
(2) does the 15+ km gap survive inside each lead-time band, and (3) does adding distance improve a
Logistic Regression that already has lead time and prior no-shows.

In [ ]:
order = ["0-7 days", "8-14 days", "15-30 days", "31-45 days", "46-60 days"]
dk = df[df["distance_known"]].copy()
dk["far"] = dk["distance_to_clinic_km"] >= 15
print("Rows with known distance:", len(dk), "of", len(df))
print("Rank correlation, distance vs lead days: %.3f" % dk["distance_to_clinic_km"].rank().corr(dk["booking_lead_days"].rank()))
print()
for b in order:
    d = dk[dk["lead_time_band"] == b]; n1, n0 = d["far"].sum(), (~d["far"]).sum()
    p1, p0 = d[d["far"]]["ns"].mean(), d[~d["far"]]["ns"].mean()
    se = np.sqrt(p1*(1-p1)/n1 + p0*(1-p0)/n0); diff = p1 - p0
    print(f"{b:11s} <15km: {p0:.1%} (n={n0})  15+km: {p1:.1%} (n={n1})  diff={diff*100:+.1f}pp  CI [{(diff-1.96*se)*100:+.1f}, {(diff+1.96*se)*100:+.1f}]")
p1, p0 = dk[dk["far"]]["ns"].mean(), dk[~dk["far"]]["ns"].mean()
print(f"ALL bands   <15km: {p0:.1%}  15+km: {p1:.1%}  diff={(p1-p0)*100:+.1f}pp")

Rows with known distance: 4910 of 5000
Rank correlation, distance vs lead days: -0.008

0-7 days    <15km: 25.5% (n=499)  15+km: 35.9% (n=128)  diff=+10.5pp  CI [+1.3, +19.6]
8-14 days   <15km: 32.8% (n=472)  15+km: 36.4% (n=118)  diff=+3.6pp  CI [-6.1, +13.3]
15-30 days  <15km: 41.4% (n=1051)  15+km: 50.2% (n=259)  diff=+8.8pp  CI [+2.0, +15.6]
31-45 days  <15km: 52.4% (n=1028)  15+km: 59.4% (n=212)  diff=+7.0pp  CI [-0.3, +14.3]
46-60 days  <15km: 66.6% (n=912)  15+km: 72.7% (n=231)  diff=+6.2pp  CI [-0.3, +12.7]
ALL bands   <15km: 47.0%  15+km: 54.1%  diff=+7.1pp


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

dk["x"] = dk["booking_lead_days"] * dk["previous_no_shows"]
base = ["booking_lead_days", "previous_no_shows", "x"]
cv5 = StratifiedKFold(5, shuffle=True, random_state=42)
for name, cols in [("without distance", base), ("with distance (km)", base + ["distance_to_clinic_km"]), ("with 15+km flag", base + ["far"])]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    sc = cross_val_score(m, dk[cols].astype(float), dk["ns"], cv=cv5, scoring="roc_auc")
    print(f"{name:20s} AUC {sc.mean():.4f} +/- {sc.std():.4f}")

without distance     AUC 0.6732 +/- 0.0101
with distance (km)   AUC 0.6772 +/- 0.0099
with 15+km flag      AUC 0.6765 +/- 0.0095


**Result: distance is not redundant with lead time, but its effect is modest.** Distance and lead
time are essentially unrelated (correlation -0.008). Inside every lead-time band the 15+ km group has
the higher no-show rate (+3.6 to +10.5 points, about +7 overall). Only two of the five bands are
individually significant, because each band has just 118 to 259 patients at 15+ km, so the
consistent direction is the finding, not any single band. Adding distance to the Logistic Regression
lifts AUC by only about 0.004, smaller than the fold-to-fold variation of about 0.01. The reason
the model gives distance little weight is the small size of the effect and the small share of
patients it applies to, not overlap with lead time. This supports keeping distance framed as a
15+ km threshold effect, not a graded factor.

## 4. Double-Risk Segment — Reproduction, Root Cause, and Stability

The Week 6 report cites a "double-risk" segment (long lead time + repeated prior no-shows) at **~4.7% of bookings, ~73% no-show risk**. First attempt at reproduction, using a lead-time threshold of 46+ days, failed to match.

In [5]:
# First attempt: lead_time >= 46 days & prior_no_show_bracket >= 2
attempt1 = df[(df["booking_lead_days"] >= 46) & (df["previous_no_shows"] >= 2)]
print(f"Attempt 1 (lead >= 46 days): n={len(attempt1)} ({len(attempt1)/len(df)*100:.2f}%), "
      f"no-show rate={attempt1['is_no_show'].mean()*100:.1f}%")
print("Does NOT match the cited 4.7% / 73% figures.\n")

# Root cause investigation: check lead_time_band boundaries
print("booking_lead_days range per lead_time_band:")
print(df.groupby("lead_time_band")["booking_lead_days"].agg(["min", "max"]))

Attempt 1 (lead >= 46 days): n=123 (2.46%), no-show rate=82.1%
Does NOT match the cited 4.7% / 73% figures.

booking_lead_days range per lead_time_band:
                min  max
lead_time_band          
0-7 days          0    7
15-30 days       15   30
31-45 days       31   45
46-60 days       46   60
8-14 days         8   14


In [6]:
# Corrected attempt: lead_time >= 31 days & prior_no_show_bracket >= 2
attempt2 = df[(df["booking_lead_days"] >= 31) & (df["previous_no_shows"] >= 2)]
print(f"Attempt 2 (lead >= 31 days): n={len(attempt2)} ({len(attempt2)/len(df)*100:.2f}%), "
      f"no-show rate={attempt2['is_no_show'].mean()*100:.1f}%")
print("Matches the cited 4.7% / 73% figures.")

Attempt 2 (lead >= 31 days): n=233 (4.66%), no-show rate=73.0%
Matches the cited 4.7% / 73% figures.


**Root cause:** the double-risk segment's lead-time threshold is 31+ days (the 31–45 and 46–60 bands combined), not 46+ days alone. Result: **FAIL → FIXED**.

**Note:** the Week 6 Decision Support report also states this flag as 46+ days. That wording is
superseded by the 31+ definition validated here.


In [7]:
# Stability check via bootstrap resampling on the corrected segment
rng = np.random.default_rng(42)
seg = attempt2["is_no_show"].values
rates = np.array([seg[rng.integers(0, len(seg), len(seg))].mean() for _ in range(2000)])

print(f"Point estimate: {seg.mean()*100:.1f}%")
print(f"Bootstrap 95% CI (2000 resamples): [{np.percentile(rates, 2.5)*100:.1f}%, {np.percentile(rates, 97.5)*100:.1f}%]")
print(f"Overall base rate for comparison: {df['is_no_show'].mean()*100:.1f}%")

Point estimate: 73.0%
Bootstrap 95% CI (2000 resamples): [67.4%, 79.0%]
Overall base rate for comparison: 48.5%


**Result: PASS.** The corrected segment is stable (tight 95% CI) and represents a genuine, substantial lift over the overall no-show base rate.

## 5. HC-POD Cross-Track Test — Data Science `double_risk_flag`

The DS track collaborator's `train_improved.py` encodes a `double_risk_flag` feature that is
documented as representing this same double-risk finding, using the threshold
`booking_lead_days >= 46`. Since that threshold was just shown to be incorrect against the
Data Analytics finding it cites, this section tests whether the discrepancy matters for the
model's performance.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

def build_features(df, threshold):
    X = df[["booking_lead_days", "previous_no_shows", "previous_appointments",
            "distance_to_clinic_km", "waiting_time_minutes"]].copy()
    cat = pd.get_dummies(df[["reminder_sent", "reminder_channel"]], drop_first=True)
    X = pd.concat([X, cat], axis=1)
    X["double_risk_flag"] = ((df["booking_lead_days"] >= threshold) & (df["previous_no_shows"] >= 2)).astype(int)
    X["leadtime_x_priornoshow"] = df["booking_lead_days"] * df["previous_no_shows"]
    X["distance_15km_plus"] = (df["distance_to_clinic_km"] >= 15).astype(int)
    return X

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for label, threshold in [("Original (>=46, as coded in train_improved.py)", 46),
                          ("Corrected (>=31, DA-validated)", 31)]:
    X = build_features(df, threshold)
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=15,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    scores = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc")
    rf.fit(X, y)
    importance = pd.Series(rf.feature_importances_, index=X.columns)["double_risk_flag"]
    results[label] = (scores.mean(), scores.std(), importance)
    print(f"{label}:")
    print(f"  CV ROC-AUC = {scores.mean():.4f} +/- {scores.std():.4f}")
    print(f"  double_risk_flag importance = {importance:.4f}\n")

Original (>=46, as coded in train_improved.py):
  CV ROC-AUC = 0.6661 +/- 0.0060
  double_risk_flag importance = 0.0223



Corrected (>=31, DA-validated):
  CV ROC-AUC = 0.6662 +/- 0.0056
  double_risk_flag importance = 0.0219



**Cross-track finding:** the original threshold (`>=46`) captures only 2.46% of bookings at
82.1% risk, versus the DA-validated 4.66% / 73.0% at the correct `>=31` threshold — silently
excluding 110 at-risk patients (62.7% no-show rate) from the flag.

**Retest result:** correcting the threshold makes **no material difference** to model
performance (ROC-AUC and feature importance are within noise of each other), because the
Random Forest already learns the interaction from the raw `booking_lead_days` and
`previous_no_shows` features.

**Validated improvement:** this is a documentation/consistency fix, not a performance fix — the
feature threshold should be corrected to `>=31` so the code, its comment, and the cited DA
source figures agree.

## 5b. Preliminary Logistic Regression Retest (superseded by the exact-candidate test)

Week 6 changed the recommended candidate model to a refined Logistic Regression, but Section 5
retested the flag threshold on the Random Forest only. The model used in this section is a simplified two-feature Logistic Regression, an early preliminary check. The exact Week 6 candidate (12 features, from `robustness_check.py`) was tested afterwards in `scripts/week7_double_risk_exact_candidate_test.py`: 0.6819 with the flag at 46, 0.6821 at 31 and 0.6820 with no flag, so the same conclusion holds. A Logistic Regression cannot learn the
lead-time by prior-no-show interaction unless it is given one, so the flag's threshold could matter
more there. Six setups are compared with 5-fold cross-validation repeated 5 times, on identical
splits, so the paired differences are meaningful.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
df["f46"] = ((df["booking_lead_days"] >= 46) & (df["previous_no_shows"] >= 2)).astype(int)
df["f31"] = ((df["booking_lead_days"] >= 31) & (df["previous_no_shows"] >= 2)).astype(int)
df["x"] = df["booking_lead_days"] * df["previous_no_shows"]
for f in ["f46", "f31"]:
    print(f"{f}: n={df[f].sum()} ({df[f].mean():.2%} of bookings), no-show rate {y[df[f]==1].mean():.1%}")
print()
raw = ["booking_lead_days", "previous_no_shows"]
sets = {"A raw only": raw, "B raw + flag46": raw + ["f46"], "C raw + flag31": raw + ["f31"],
        "D raw + interaction": raw + ["x"], "E raw + interaction + flag46": raw + ["x", "f46"], "F raw + interaction + flag31": raw + ["x", "f31"]}
cvr = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
res = {}
for k, cols in sets.items():
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    res[k] = cross_val_score(m, df[cols].astype(float), y, cv=cvr, scoring="roc_auc")
    print(f"{k:30s} AUC {res[k].mean():.4f} +/- {res[k].std():.4f}")
print()
for a, b in [("C raw + flag31", "B raw + flag46"), ("F raw + interaction + flag31", "E raw + interaction + flag46")]:
    d = res[a] - res[b]
    print(f"paired diff [{a}] minus [{b}]: {d.mean():+.4f} (sd {d.std():.4f}, 31+ wins {int((d>0).sum())}/{len(d)} folds)")

f46: n=123 (2.46% of bookings), no-show rate 82.1%
f31: n=233 (4.66% of bookings), no-show rate 73.0%

A raw only                     AUC 0.6719 +/- 0.0130
B raw + flag46                 AUC 0.6719 +/- 0.0129
C raw + flag31                 AUC 0.6717 +/- 0.0129
D raw + interaction            AUC 0.6717 +/- 0.0129
E raw + interaction + flag46   AUC 0.6717 +/- 0.0129
F raw + interaction + flag31   AUC 0.6718 +/- 0.0126

paired diff [C raw + flag31] minus [B raw + flag46]: -0.0001 (sd 0.0006, 31+ wins 10/25 folds)
paired diff [F raw + interaction + flag31] minus [E raw + interaction + flag46]: +0.0000 (sd 0.0009, 31+ wins 16/25 folds)


**Result: the threshold makes no measurable difference to the Logistic Regression either.** The 31+
minus 46+ AUC difference is -0.0001 (31+ wins in 10 of 25 folds), and it stays near zero when the
interaction term is included. The flag covers only 2.5% to 4.7% of bookings, so it cannot move overall
AUC. It works as a business rule for deciding who gets a call, not as a model feature. On that basis
**the 31+ definition is kept**: it reaches 233 patients at a 73.0% no-show rate, against 123 patients at
82.1% for 46+, so it covers nearly twice as many high-risk patients at a still very high rate.

**Exact Week 6 candidate (standalone script):** 0.6819 for 46+ vs 0.6821 for 31+, and 0.6820 with no flag at all, so the conclusion is unchanged. That test uses the 4,737 attended and no-show bookings the Week 6 model is trained on, where the counts are 225 patients (75.6%) for 31+ against 120 (84.2%) for 46+; the 233 and 123 above are on all 5,000 bookings.

## 6. Summary

| Test | Result |
|---|---|
| KPI 1-4 reconciliation (dashboard/CSV/recompute) | PASS |
| KPI 5 (cancellation-to-no-show ratio) reproduction | PASS |
| Lead-time finding across appointment_type | PASS |
| Lead-time finding across age_group | PASS |
| Distance-band CI overlap check | PASS |
| Distance vs lead time redundancy (in-band gap, correlation, model AUC) | Not redundant; effect real but modest |
| Double-risk segment reproduction (first attempt) | FAIL, root-caused and fixed |
| Double-risk segment stability (bootstrap) | PASS |
| Cross-track `double_risk_flag` threshold check (Random Forest) | Discrepancy found, quantified, retested, corrected |
| Cross-track `double_risk_flag` threshold check (exact Week 6 candidate Logistic Regression, standalone script) | No difference; 31+ kept |

See the accompanying PDF report for the full narrative, dashboard testing evidence, business
recommendations, limitations, and the Week 8 pilot design, and the Week 7 Project Summary PDF.